# Notebook 2 — Data Type Handling
### Sprint 5 | Data Cleaning & Preprocessing for AI/ML Engineers

**Methodology:** Understand → Demonstrate → Implement, for every topic, grounded in
the Telco Customer Churn dataset carried over from Sprint 4.


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("telco_churn.csv")
print(f"Dataset loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")


Dataset loaded: 7,043 rows, 21 columns


---
## 1. Identifying Data Types

### Understand
Every column has a stored dtype, but the stored dtype and the column's *true* nature
aren't always the same thing — this mismatch is exactly what this whole notebook exists
to catch and fix.

### Demonstrate
**Business example:** `SeniorCitizen` is stored as `int64` (0/1), but it's genuinely a
categorical flag, not a quantity — "1.5 senior citizen" is meaningless, unlike a real
integer such as a count.

### Implement


In [2]:
print(df.dtypes)
print(f"\n'SeniorCitizen' unique values: {sorted(df['SeniorCitizen'].unique())} -> binary flag, not a true integer quantity")


customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

'SeniorCitizen' unique values: [np.int64(0), np.int64(1)] -> binary flag, not a true integer quantity


**Finding:** `TotalCharges` is `object` (text) despite representing currency — the
main problem this notebook resolves. `SeniorCitizen` is `int64` but semantically
categorical — a judgment call worth documenting even though no code change is strictly
required for it to function in most models.


---
## 2. Numerical, 3. Categorical, 4. Boolean, 5. Date/Time, 6. String Data

### Understand
Every column falls into one of these natural-language categories, independent of its
Pandas dtype: **numerical** (arithmetic is meaningful), **categorical** (a label from a
fixed set), **boolean** (strictly two logical states), **date/time** (a calendar
point), or **string** (free text). Correctly classifying each column is the prerequisite
for choosing the right preprocessing technique later in this sprint.

### Demonstrate
**AI/ML use case:** A categorical column gets one-hot encoded (Notebook 7); a numerical
column gets scaled (Notebook 8); treating one as the other produces nonsense — e.g.,
one-hot encoding `MonthlyCharges` would create thousands of near-useless columns.

### Implement


In [3]:
column_classification = {
    'Numerical': ['tenure', 'MonthlyCharges', 'TotalCharges'],
    'Categorical (nominal)': ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
                               'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                               'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling',
                               'PaymentMethod', 'Churn'],
    'Categorical (ordinal)': ['Contract'],   # Month-to-month < One year < Two year has a natural order
    'Boolean-like (0/1)': ['SeniorCitizen'],
    'Date/Time': [],   # none present in this dataset
    'Identifier (string)': ['customerID'],
}
for category, cols in column_classification.items():
    print(f"{category} ({len(cols)}): {cols}")


Numerical (3): ['tenure', 'MonthlyCharges', 'TotalCharges']
Categorical (nominal) (15): ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'PaymentMethod', 'Churn']
Categorical (ordinal) (1): ['Contract']
Boolean-like (0/1) (1): ['SeniorCitizen']
Date/Time (0): []
Identifier (string) (1): ['customerID']


**Finding:** This dataset has **no date/time column at all** — every record is a
single snapshot, not a time series (confirmed already in Sprint 4, Notebook 2). `Contract`
is the only column with a genuine, business-meaningful order (Month-to-month < One year <
Two year), making it a candidate for ordinal encoding rather than one-hot in Notebook 7.


---
## 7. Converting Data Types — `astype()`

### Understand
`.astype()` directly forces a column to a new dtype — fast and simple, but it raises an
error (rather than silently coping) if any value genuinely can't convert.

### Demonstrate
**Simple example:** `SeniorCitizen` converted from `int64` to a proper categorical dtype,
formalizing the judgment call made in Topic 1.

### Implement


In [4]:
print(f"Before: SeniorCitizen dtype = {df['SeniorCitizen'].dtype}")
df['SeniorCitizen'] = df['SeniorCitizen'].astype('category')
print(f"After : SeniorCitizen dtype = {df['SeniorCitizen'].dtype}")

# Demonstrating astype()'s failure mode on purpose
try:
    df['MonthlyCharges'].astype('int64').astype('str').astype('int64')  # fine, all-numeric text
    bad_series = pd.Series(['12', '15', 'not_a_number'])
    bad_series.astype('int64')
except ValueError as e:
    print(f"\nastype() FAILS loudly on non-numeric text: {type(e).__name__}: {str(e)[:100]}")


Before: SeniorCitizen dtype = int64
After : SeniorCitizen dtype = category

astype() FAILS loudly on non-numeric text: ValueError: invalid literal for int() with base 10: 'not_a_number'


**Finding:** `.astype()` succeeded instantly for `SeniorCitizen` since every value
was already numeric-compatible. The deliberate failure demo shows why `.astype()` is
risky for messy real-world text — it has no "safe" mode, unlike `pd.to_numeric()`
(Topic 8), which is why `TotalCharges` uses that function instead, next.


---
## 8. Converting Data Types — `to_numeric()`

### Understand
`pd.to_numeric(..., errors='coerce')` converts a column to numeric, turning any
un-convertible value into `NaN` instead of crashing — the safe choice for messy,
real-world text-typed numeric columns.

### Demonstrate
**Business example:** `TotalCharges` has 11 blank-string entries mixed in with genuinely
numeric text — exactly the scenario `to_numeric(errors='coerce')` is built for.

### Implement


In [5]:
print(f"Before: dtype={df['TotalCharges'].dtype}, blank entries={(df['TotalCharges'].str.strip()=='').sum()}")

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print(f"After : dtype={df['TotalCharges'].dtype}, NaN entries={df['TotalCharges'].isnull().sum()}")


Before: dtype=str, blank entries=11
After : dtype=float64, NaN entries=11


**Finding:** The conversion succeeds cleanly, and the 11 previously-invisible blanks
become explicit, correctly-typed `NaN` values — ready for Notebook 3's missing-value
treatment. **This is the single most important data-type fix in this entire sprint.**


---
## 9. Converting Data Types — `to_datetime()`

### Understand
`pd.to_datetime()` converts text into a proper datetime type (Sprint 3, Notebook 11) —
this dataset has no date column, so this topic is demonstrated on a small illustrative
example rather than forced onto data that doesn't have one.

### Demonstrate
**AI/ML use case:** If this company's data instead included a `SignupDate` column (a
realistic field many real telecom datasets do have), it would need exactly this
conversion before any tenure-related date arithmetic could be trusted.

### Implement


In [6]:
illustrative_dates = pd.Series(['2024-01-15', '2024-02-20', '2024-03-05'])
converted = pd.to_datetime(illustrative_dates)
print("Before:", illustrative_dates.tolist(), "dtype:", illustrative_dates.dtype)
print("After :", converted.tolist(), "dtype:", converted.dtype)


Before: ['2024-01-15', '2024-02-20', '2024-03-05'] dtype: str
After : [Timestamp('2024-01-15 00:00:00'), Timestamp('2024-02-20 00:00:00'), Timestamp('2024-03-05 00:00:00')] dtype: datetime64[us]


**Finding:** Not applicable to this specific dataset (no date columns exist,
confirmed in Sprint 4), demonstrated illustratively instead. **Documented honestly as
"not applicable" rather than fabricated** — a real, useful negative finding in itself.


---
## 10. Handling Invalid Conversions

### Understand
A "invalid conversion" happens when a value can't honestly become the target type —
`to_numeric`/`to_datetime` handle this safely with `errors='coerce'`; `.astype()` does
not, and will raise instead.

### Demonstrate
**Business example:** If a data-entry error put the text "unknown" into `MonthlyCharges`
for one row, `to_numeric(errors='coerce')` would turn it into `NaN` (fixable later),
while `.astype(float)` would crash the whole script.

### Implement


In [7]:
messy_charges = pd.Series(['45.50', '60.00', 'unknown', '32.10'])
safe_conversion = pd.to_numeric(messy_charges, errors='coerce')
print("Safe conversion (errors='coerce'):")
print(safe_conversion)
print(f"\nRow(s) needing follow-up: {safe_conversion.isnull().sum()}")


Safe conversion (errors='coerce'):
0    45.5
1    60.0
2     NaN
3    32.1
dtype: float64

Row(s) needing follow-up: 1


**Finding:** `errors='coerce'` isolates exactly one bad row as `NaN`, letting the
rest of the column convert successfully — this is the standard pattern for cleaning any
numeric-looking text column without losing the whole column to one bad value.


---
## 11. Detecting Incorrect Data Types

### Understand
A systematic check — comparing each column's *stored* dtype against what it *should*
logically be — catches issues like `TotalCharges` before they cause downstream failures.

### Demonstrate
**AI/ML use case:** Running this check immediately after loading any new dataset is a
cheap, high-value habit (Sprint 4, Notebook 2's `.dtypes` check, now formalized as a rule
engine).

### Implement


In [8]:
def detect_type_issues(data):
    issues = []
    for col in data.select_dtypes(include='object').columns:
        sample = data[col].dropna().head(50)
        numeric_convertible = pd.to_numeric(sample, errors='coerce').notna().mean()
        if numeric_convertible > 0.9 and col != 'customerID':
            issues.append(f"{col}: {numeric_convertible*100:.0f}% of values look numeric but column is stored as text")
    return issues

df_raw = pd.read_csv("telco_churn.csv")
for issue in detect_type_issues(df_raw):
    print(issue)


TotalCharges: 100% of values look numeric but column is stored as text


/tmp/ipykernel_499/2310374976.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in data.select_dtypes(include='object').columns:


**Finding:** The detector correctly flags `TotalCharges` as the one column stored
incorrectly — the exact same issue found manually in Sprint 4, now caught by an automated
rule rather than visual inspection, demonstrating how this check could scale to a much
wider dataset.


---
## 12. Worked Examples: String→Numeric, String→Date, Numeric→Categorical, Integer→Float

### Implement


In [9]:
# String -> Numeric (the real fix already applied above)
print("String -> Numeric: TotalCharges is now", df['TotalCharges'].dtype)

# String -> Date (illustrative, since no date column exists)
print("String -> Date (illustrative):", pd.to_datetime(['2024-06-01']).dtype)

# Numeric -> Categorical
print("Numeric -> Categorical: SeniorCitizen is now", df['SeniorCitizen'].dtype)

# Integer -> Float
tenure_as_float = df['tenure'].astype('float64')
print(f"Integer -> Float: tenure {df['tenure'].dtype} -> {tenure_as_float.dtype}")


String -> Numeric: TotalCharges is now float64
String -> Date (illustrative): datetime64[us]
Numeric -> Categorical: SeniorCitizen is now category
Integer -> Float: tenure int64 -> float64


### Why Correct Data Types Matter

Every downstream step in this sprint depends on types being right first: missing-value
detection (Notebook 3) only works correctly on a truly-numeric `TotalCharges`; scaling
(Notebook 8) requires numeric input; encoding (Notebook 7) requires categorical columns to
actually be recognized as such. A single wrong dtype early in the pipeline propagates
errors — or worse, silent wrong behavior — through every later stage.


---
## Summary

| Topic | Finding for THIS dataset |
|---|---|
| Identifying types | `TotalCharges` (object, should be float), `SeniorCitizen` (int, semantically categorical) |
| astype() | Fast but fails loudly on non-numeric text — demonstrated deliberately |
| to_numeric() | The correct fix for `TotalCharges`; converts blanks to real NaN safely |
| to_datetime() | Not applicable — no date columns exist in this dataset |
| Invalid conversions | `errors='coerce'` isolates bad values instead of crashing the whole column |
| Detection | An automated rule-based scan correctly flags `TotalCharges` |

**Next notebook:** `03_Missing_Value_Handling.ipynb` — now that `TotalCharges`'s 11
missing values are properly visible as `NaN`, this notebook decides how to handle them.
